In [7]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from google.colab import files


# Upload files manually

uploaded = files.upload()  # upload set_1.class and set_2.class

def pick_uploaded(prefix, ext=".class"):
    matches = [f for f in uploaded.keys() if f.startswith(prefix) and f.endswith(ext)]
    if not matches:
        raise FileNotFoundError(f"No {prefix}*.class found. Uploaded: {list(uploaded.keys())}")
    return sorted(matches, key=len)[0]

FILE1 = pick_uploaded("set_1")
FILE2 = pick_uploaded("set_2")


# Parse .class (assumes: ID label full_evalue best_domain_evalue)
# We'll use FULL-SEQUENCE E-value (column index 2)
LABEL_COL = 1
EV_COL = 2  # full-seq e-value

def load_data(fn):
    text = uploaded[fn].decode("utf-8", errors="replace")
    out = []
    for line in text.splitlines():
        if not line.strip() or line.startswith("#"):
            continue
        parts = line.split()
        if len(parts) <= max(LABEL_COL, EV_COL):
            continue
        y = int(parts[LABEL_COL])
        e = float(parts[EV_COL])
        out.append((y, e))
    return out

def confusion_counts(data, thr):
    TP = FP = TN = FN = 0
    for y, e in data:
        pred = 1 if e <= thr else 0
        if pred == 1 and y == 1: TP += 1
        elif pred == 1 and y == 0: FP += 1
        elif pred == 0 and y == 0: TN += 1
        else: FN += 1
    return TN, FP, FN, TP

data1 = load_data(FILE1)
data2 = load_data(FILE2)

# thresholds
thresholds = [0.001, 0.002]

results = {}
for t in thresholds:
    results[(t, "I Dataset")]  = confusion_counts(data1, t)
    results[(t, "II Dataset")] = confusion_counts(data2, t)


# Drawing helpers

def draw_fold_box(ax, x0, y0, w, h, title, TN, FP, FN, TP):
    # outer box
    ax.add_patch(Rectangle((x0, y0), w, h, fill=False, linewidth=1.2))

    # split into 2 cols, 2 rows
    ax.plot([x0 + w/2, x0 + w/2], [y0, y0 + h], color="black", linewidth=1.0)
    ax.plot([x0, x0 + w], [y0 + h/2, y0 + h/2], color="black", linewidth=1.0)

    # title above the box
    ax.text(x0 + w/2, y0 + h + 0.035, title, ha="center", va="bottom", fontsize=11)

    # labels (top-left, top-right, bottom-left, bottom-right)
    ax.text(x0 + w*0.05, y0 + h*0.75, "True Negatives", ha="left", va="center", fontsize=10)
    ax.text(x0 + w*0.55, y0 + h*0.75, "False Positives", ha="left", va="center", fontsize=10)
    ax.text(x0 + w*0.05, y0 + h*0.25, "False Negatives", ha="left", va="center", fontsize=10)
    ax.text(x0 + w*0.55, y0 + h*0.25, "True Positives", ha="left", va="center", fontsize=10, fontweight="bold")

    # numbers (bottom-left area of each cell)
    ax.text(x0 + w*0.05, y0 + h*0.60, f"{TN}", ha="left", va="center", fontsize=10)
    ax.text(x0 + w*0.55, y0 + h*0.60, f"{FP}", ha="left", va="center", fontsize=10)
    ax.text(x0 + w*0.05, y0 + h*0.10, f"{FN}", ha="left", va="center", fontsize=10)
    ax.text(x0 + w*0.55, y0 + h*0.10, f"{TP}", ha="left", va="center", fontsize=10)

def draw_threshold_block(ax, y_top, block_title, t):
    ax.text(0.0, y_top, block_title, ha="left", va="top", fontsize=13, fontweight="bold")

    # geometry
    box_w, box_h = 0.46, 0.20
    gap_x = 0.04
    y0 = y_top - 0.28
    x_left = 0.04
    x_right = x_left + box_w + gap_x

    TN1, FP1, FN1, TP1 = results[(t, "I Dataset")]
    TN2, FP2, FN2, TP2 = results[(t, "II Dataset")]

    draw_fold_box(ax, x_left,  y0, box_w, box_h, "I Dataset",  TN1, FP1, FN1, TP1)
    draw_fold_box(ax, x_right, y0, box_w, box_h, "II Dataset", TN2, FP2, FN2, TP2)

# Make the final figure

fig = plt.figure(figsize=(11, 7))
ax = fig.add_axes([0, 0, 1, 1])
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

draw_threshold_block(ax, 0.95, "Table 1: 2 fold cross validation - confusion matrices, threshold 0.001", 0.001)
draw_threshold_block(ax, 0.48, "Table 2: 2 fold cross validation - confusion matrices, threshold 0.002", 0.002)

# caption
ax.text(0.5, 0.06, "Figure X: confusion matrices for the threshold values of 0.001 and 0.002",
        ha="center", va="center", fontsize=14, style="italic")

out = "confusion_matrices_0p001_0p002.png"
plt.savefig(out, dpi=300, bbox_inches="tight")
plt.close()
print("Saved:", out)

files.download(out)


Saving set_1.class to set_1 (6).class
Saving set_2.class to set_2 (6).class
Saved: confusion_matrices_0p001_0p002.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>